# Market Access in Interwar Poland

This notebook keeps only the components needed for the final empirical results.

What is moved out to `market_access_helpers.py`:
- IO and parsing,
- scenario/year discovery,
- population preprocessing,
- export and plotting helpers.

What remains in this notebook:
- core economic formulas,
- market-access assembly logic,
- scenario execution and validation checks.


## 1) Setup and constants

We compute annual market access for years 1924-1938, for two scenarios:
- `baseline` (distance from horse+rail km matrices),
- `fixed14` (distance from fixed14 time matrices).

Estimated parameters used here:
- domestic distance coefficient: `-2.6705`,
- foreign distance coefficient: `-0.5684`,
- annual partition coefficient from `partition_coefficients.csv`.


In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

os.chdir("../../..")
sys.path.append(str(Path.cwd() / "examples" / "interwar_poland"))
sys.path.append(str(Path.cwd() / "examples" / "interwar_poland" / "market_access"))

from global_definitions import adm_history_plotter, d_city_mapping
from market_access_helpers import (
    border_id_norm,
    build_border_connections,
    build_foreign_gdp_long,
    build_point_population,
    collect_target_pair_distances,
    collect_foreign_city_route_distances,
    build_population_inputs,
    list_available_years,
    load_distance_matrix_for_scenario,
    load_partition_coefficients,
    load_partition_dummies,
    normalize_text,
    save_annual_maps,
    save_change_map_1938_vs_1924,
    save_tables,
)

BASE_DIR = Path("examples/interwar_poland/market_access")
DATA_DIR = BASE_DIR / "data"
DIST_DIR = DATA_DIR / "distances"
OUT_DIR = BASE_DIR / "outputs" / "market_access"
PLOTS_DIR = BASE_DIR / "plots" / "new_formula"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

DISTRICTS_GEOJSON = DATA_DIR / "districts_1934_10_1.geojson"
CITY_POP_CSV = DATA_DIR / "city_population.csv"
RURAL_POP_CSV = DATA_DIR / "rural_population.csv"
PARTITION_DUMMIES_CSV = DATA_DIR / "partition_dummies.csv"
PARTITION_COEFF_CSV = DATA_DIR / "partition_coefficients.csv"
FOREIGN_GDP_CSV = DATA_DIR / "foreign_region_gdp.csv"
BORDER_CONN_CSV = DATA_DIR / "border_crossing_IIRP_connections.csv"
CITY_COORDS_GEOJSON = Path("data/adm_histories/interwar_poland/cities_coords/cities_coords.geojson")

YEARS_TARGET = list(range(1924, 1939))
SCENARIOS = ["baseline", "fixed14"]
ADM_STATE_DATE = datetime(1934, 10, 1)

COEFF_DISTANCE_DOMESTIC = -2.6705
COEFF_DISTANCE_FOREIGN = -0.5684
EPS_DISTANCE = 1e-9

# Unit switch: convert all km-based distances to miles when True.
COMPUTE_IN_MILES = True
KM_TO_MILES = 0.621371

# For fixed14 we need an external connector speed in matching units.
FIXED14_EXTERNAL_SPEED_KMPH = 14.0
FIXED14_EXTERNAL_SPEED_MPH = FIXED14_EXTERNAL_SPEED_KMPH * KM_TO_MILES


Loading changes list...
✅ Loaded 309 validated changes in 0.05 seconds.
Loading initial state...
✅ Loaded initial state.
Loading initial district registry...
✅ Loaded 292 validated districts in 0.03 seconds. Set their initial state timespans to (1921-02-19, 1939-09-01).
Loading initial region registry...
✅ Loaded 19 validated regions in 0.00 seconds. Set their initial state timespans to (1921-02-19, 1939-09-01)
Creating administrative history (sequentially applying changes)...
✅ Successfully applied all changes in 11.90 seconds. Administrative history database created.
Loading territories...
Loaded: districts_1922_generalized_dp_800m.shp (271 rows)
Loaded: districts_1929_generalized_dp_800m.shp (281 rows)
Loaded: districts_1931_generalized_dp_800m.shp (283 rows)
Loaded: districts_1934_generalized_dp_800m.shp (264 rows)
Loaded: districts_1939_generalized_dp_800m.shp (265 rows)
✅ Successfully loaded all territories in 1.02 seconds.
Deducing all possible dist territories on the basis of t

## 2) Load static datasets

`District` is the canonical county identifier used throughout joins.


In [2]:
partition_coeff = load_partition_coefficients(PARTITION_COEFF_CSV)
partition_df = load_partition_dummies(PARTITION_DUMMIES_CSV)
foreign_gdp_long = build_foreign_gdp_long(FOREIGN_GDP_CSV, YEARS_TARGET)
border_conn = build_border_connections(BORDER_CONN_CSV)

districts_gdf, district_list, city_keep, rural_keep = build_population_inputs(
    districts_geojson=DISTRICTS_GEOJSON,
    city_coords_geojson=CITY_COORDS_GEOJSON,
    city_pop_csv=CITY_POP_CSV,
    rural_pop_csv=RURAL_POP_CSV,
    years_target=YEARS_TARGET,
)

print("Districts:", len(district_list))
print("Partition coeff years:", min(partition_coeff), "-", max(partition_coeff))
print("Scenarios:", SCENARIOS)


Districts: 247
Partition coeff years: 1924 - 1938
Scenarios: ['baseline', 'fixed14']


## 3) Core economic building blocks

This section contains the key analytical functions:
- population-weighted district-to-district distance aggregation,
- partition-border dummy by partition difference,
- domestic MA contribution,
- foreign MA contribution.


In [3]:
def compute_district_distance_matrix(dm: pd.DataFrame, point_pop: pd.DataFrame) -> pd.DataFrame:
    # Collapse point-level distances into district-level weighted means.
    pp = point_pop[["point_id", "District", "pop"]].copy()

    x = dm.merge(pp, left_on="origin_id", right_on="point_id", how="left").rename(
        columns={"District": "origin_district", "pop": "origin_pop"}
    )
    x = x.merge(pp, left_on="dest_id", right_on="point_id", how="left", suffixes=("", "_dest")).rename(
        columns={"District": "dest_district", "pop": "dest_pop"}
    )

    x = x.dropna(subset=["origin_district", "dest_district"]).copy()
    x = x[x["origin_district"] != x["dest_district"]].copy()

    x["distance_value"] = pd.to_numeric(x["distance_value"], errors="coerce")
    x = x[x["distance_value"].notna()].copy()

    x["w"] = x["origin_pop"].fillna(0.0) * x["dest_pop"].fillna(0.0)
    x = x[x["w"] > 0].copy()
    x["wd"] = x["w"] * x["distance_value"]

    g = x.groupby(["origin_district", "dest_district"], as_index=False).agg(w_sum=("w", "sum"), wd_sum=("wd", "sum"))
    g["distance_value"] = g["wd_sum"] / g["w_sum"]
    return g[["origin_district", "dest_district", "distance_value"]].copy()


def compute_district_to_border_distance(dm: pd.DataFrame, point_pop: pd.DataFrame) -> pd.DataFrame:
    # For each district and border crossing, compute weighted mean distance from district points.
    pp = point_pop[["point_id", "District", "pop"]].copy()

    x = dm.merge(pp, left_on="origin_id", right_on="point_id", how="left")
    x = x.rename(columns={"District": "origin_district", "pop": "origin_pop"})

    x = x[x["dest_id"].astype(str).str.startswith("Border_Crossing:")].copy()
    x = x.dropna(subset=["origin_district"]).copy()

    x["distance_value"] = pd.to_numeric(x["distance_value"], errors="coerce")
    x["origin_pop"] = pd.to_numeric(x["origin_pop"], errors="coerce").fillna(0.0)
    x = x[x["distance_value"].notna() & (x["origin_pop"] > 0)].copy()
    x["wd"] = x["origin_pop"] * x["distance_value"]

    g = x.groupby(["origin_district", "dest_id"], as_index=False).agg(w_sum=("origin_pop", "sum"), wd_sum=("wd", "sum"))
    g["distance_to_border"] = g["wd_sum"] / g["w_sum"]
    return g[["origin_district", "dest_id", "distance_to_border"]].copy()


def compute_partition_pair_dummy(district_distances: pd.DataFrame, partition_lookup: pd.DataFrame) -> pd.Series:
    # Binary dummy: 1 if origin and destination belonged to different historic partitions.
    part_map = dict(zip(partition_lookup["District"], partition_lookup["partition_label"]))
    o = district_distances["origin_district"].map(part_map).fillna("UNKNOWN")
    d = district_distances["dest_district"].map(part_map).fillna("UNKNOWN")
    return (o != d).astype(int)


def compute_domestic_ma(
    district_distances: pd.DataFrame,
    district_mass: pd.DataFrame,
    partition_lookup: pd.DataFrame,
    coeff_part_border_year: float,
) -> pd.DataFrame:
    # Within-Poland MA component.
    mass_map = dict(zip(district_mass["District"], district_mass["mass"]))

    x = district_distances.copy()
    x["dest_mass"] = x["dest_district"].map(mass_map).fillna(0.0)
    x["part_dummy"] = compute_partition_pair_dummy(x, partition_lookup)

    x["contrib_domestic"] = x["dest_mass"] * np.exp(
        COEFF_DISTANCE_DOMESTIC * np.log(x["distance_value"] + EPS_DISTANCE)
        + coeff_part_border_year * x["part_dummy"]
    )

    out = x.groupby("origin_district", as_index=False)["contrib_domestic"].sum().rename(
        columns={"origin_district": "District", "contrib_domestic": "ma_domestic"}
    )
    return out


def compute_foreign_ma(
    year: int,
    scenario: str,
    district_to_border: pd.DataFrame,
    border_connections: pd.DataFrame,
    foreign_gdp: pd.DataFrame,
) -> pd.DataFrame:
    # Foreign MA component through mapped border crossings and external connector lengths.
    bd = district_to_border.copy()
    bd["border_norm"] = bd["dest_id"].map(border_id_norm)

    conn = border_connections.copy()
    merged = bd.merge(conn, left_on="border_norm", right_on="border_crossing_norm", how="inner")
    merged["region_norm"] = merged["country_or_province"].map(normalize_text)

    if scenario == "fixed14":
        connector_length = merged["length_km"] * (KM_TO_MILES if COMPUTE_IN_MILES else 1.0)
        speed = FIXED14_EXTERNAL_SPEED_MPH if COMPUTE_IN_MILES else FIXED14_EXTERNAL_SPEED_KMPH
        connector_distance = connector_length * (60.0 / speed)
    else:
        connector_distance = merged["length_km"] * (KM_TO_MILES if COMPUTE_IN_MILES else 1.0)

    merged["distance_to_foreign"] = merged["distance_to_border"] + connector_distance

    # If multiple routes map to one region, keep the shortest effective route.
    merged = merged.groupby(["origin_district", "region_norm"], as_index=False)["distance_to_foreign"].min()

    gdp_y = foreign_gdp[foreign_gdp["year"] == year][["region_norm", "gdp"]].copy()
    x = merged.merge(gdp_y, on="region_norm", how="inner")

    x["contrib_foreign"] = x["gdp"] * np.exp(COEFF_DISTANCE_FOREIGN * np.log(x["distance_to_foreign"] + EPS_DISTANCE))

    out = x.groupby("origin_district", as_index=False)["contrib_foreign"].sum().rename(
        columns={"origin_district": "District", "contrib_foreign": "ma_foreign"}
    )
    return out


## 4) Scenario runner

For each available year in a scenario:
1. Load distance matrix,
2. Build district masses from rural + city populations,
3. Compute domestic MA and foreign MA,
4. Store total + decomposition.


In [4]:
def run_scenario(scenario: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    years = list_available_years(DIST_DIR, scenario, YEARS_TARGET)
    print(f"Scenario={scenario}; years found: {years}")
    if not years:
        empty_ma = pd.DataFrame(columns=["District", "year", "scenario", "ma_total", "ma_domestic", "ma_foreign"])
        empty_dist = pd.DataFrame(columns=["scenario", "year", "origin_district", "dest_district", "distance_value"])
        empty_border = pd.DataFrame(columns=["scenario", "year", "origin_district", "dest_id", "distance_to_border"])
        return empty_ma, empty_dist, empty_border

    rows_ma = []
    rows_dist = []
    rows_border = []

    for year in years:
        print("  year", year)

        dm = load_distance_matrix_for_scenario(
            DIST_DIR,
            year,
            scenario,
            compute_in_miles=COMPUTE_IN_MILES,
        )
        point_pop, district_mass = build_point_population(year, city_keep, rural_keep)

        district_distances = compute_district_distance_matrix(dm, point_pop)
        district_distances = district_distances.copy()
        district_distances["scenario"] = scenario
        district_distances["year"] = year
        rows_dist.append(
            district_distances[["scenario", "year", "origin_district", "dest_district", "distance_value"]]
        )

        district_to_border = compute_district_to_border_distance(dm, point_pop)
        district_to_border = district_to_border.copy()
        district_to_border["scenario"] = scenario
        district_to_border["year"] = year
        rows_border.append(
            district_to_border[["scenario", "year", "origin_district", "dest_id", "distance_to_border"]]
        )

        coeff_part = partition_coeff.get(year)
        if coeff_part is None:
            raise RuntimeError(f"Missing partition coefficient for {year}")

        dom = compute_domestic_ma(district_distances, district_mass, partition_df, coeff_part)
        foreign = compute_foreign_ma(year, scenario, district_to_border, border_conn, foreign_gdp_long)

        out = district_mass[["District"]].copy()
        out["year"] = year
        out["scenario"] = scenario
        out = out.merge(dom, on="District", how="left")
        out = out.merge(foreign, on="District", how="left")

        out["ma_domestic"] = out["ma_domestic"].fillna(0.0)
        out["ma_foreign"] = out["ma_foreign"].fillna(0.0)
        out["ma_total"] = out["ma_domestic"] + out["ma_foreign"]

        rows_ma.append(out[["District", "year", "scenario", "ma_total", "ma_domestic", "ma_foreign"]])

    ma_df = pd.concat(rows_ma, ignore_index=True)
    dist_df = pd.concat(rows_dist, ignore_index=True)
    border_df = pd.concat(rows_border, ignore_index=True)
    return ma_df, dist_df, border_df


scenario_results = [run_scenario(sc) for sc in SCENARIOS]
ma_all = pd.concat([r[0] for r in scenario_results], ignore_index=True)
district_distances_all = pd.concat([r[1] for r in scenario_results], ignore_index=True)
district_to_border_all = pd.concat([r[2] for r in scenario_results], ignore_index=True)

print("Rows computed (ma_all):", len(ma_all))
print("Rows computed (district_distances_all):", len(district_distances_all))
print("Rows computed (district_to_border_all):", len(district_to_border_all))
ma_all.head()


Scenario=baseline; years found: [1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938]
  year 1924
  year 1925
  year 1926
  year 1927
  year 1928
  year 1929
  year 1930
  year 1931
  year 1932
  year 1933
  year 1934
  year 1935
  year 1936
  year 1937
  year 1938
Scenario=fixed14; years found: [1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938]
  year 1924
  year 1925
  year 1926
  year 1927
  year 1928
  year 1929
  year 1930
  year 1931
  year 1932
  year 1933
  year 1934
  year 1935
  year 1936
  year 1937
  year 1938
Rows computed (ma_all): 7410
Rows computed (district_distances_all): 1822860
Rows computed (district_to_border_all): 37050


,District,year,scenario,ma_total,ma_domestic,ma_foreign
0,KOŚCIAŃSKI,1924,baseline,96.445875,96.445875,0.0
1,ZBOROWSKI,1924,baseline,59.547995,59.547995,0.0
2,MIĘDZYCHODZKI,1924,baseline,27.462675,27.462675,0.0
3,KOŁOMYJSKI,1924,baseline,67.083146,67.083146,0.0
4,DZIŚNIEŃSKI,1924,baseline,10.987892,10.987892,0.0


## 5) Quick validation checks

These checks are intentionally simple and transparent for auditability.


In [5]:
if ma_all.empty:
    print("No results computed yet (missing matrices in target years).")
else:
    coverage = ma_all.groupby(["scenario", "year"], as_index=False).agg(
        districts=("District", "nunique"),
        ma_total_min=("ma_total", "min"),
        ma_total_median=("ma_total", "median"),
        ma_total_max=("ma_total", "max"),
    )
    display(coverage)

    check_add = (ma_all["ma_total"] - (ma_all["ma_domestic"] + ma_all["ma_foreign"]))
    print("Max decomposition error:", float(np.abs(check_add).max()))


,scenario,year,districts,ma_total_min,ma_total_median,ma_total_max
0,baseline,1924,247,4.962370,53.930629,4451.407598
1,baseline,1925,247,5.094287,55.850119,4448.113927
2,baseline,1926,247,5.227462,58.619365,4445.023028
3,baseline,1927,247,5.360216,59.629327,4442.117027
4,baseline,1928,247,5.494084,60.476881,4439.397389
5,baseline,1929,247,5.620996,61.336294,4436.853725
6,baseline,1930,247,5.752748,62.240233,4434.541263
7,baseline,1931,247,5.891488,63.282326,4432.381132
8,baseline,1932,247,6.021594,64.993180,7868.087959
9,baseline,1933,247,6.146957,67.531328,4425.274800


Max decomposition error: 0.0


In [ ]:
TARGET_DISTANCE_PAIRS = [
    ("M. ST. WARSZAWA", "KRAKÓW (MIASTO)"),
    ("M. ST. WARSZAWA", "WILNO (MIASTO)"),
    ("M. ST. WARSZAWA", "LWÓW (MIASTO)"),
    ("M. ST. WARSZAWA", "BIAŁOSTOCKI"),
    ("M. ST. WARSZAWA", "MORSKI"),
    ("BIAŁOSTOCKI", "WILNO (MIASTO)"),
    ("KRAKÓW (MIASTO)", "KIELECKI"),
    ("BIAŁOSTOCKI", "LWÓW (MIASTO)"),
    ("MORSKI", "BIELSKO"),
    ("POZNAŃ (MIASTO)", "BIELSKO"),
]


distance_check = collect_target_pair_distances(
    ma_all=ma_all,
    district_distances_all=district_distances_all,
    compute_in_miles=COMPUTE_IN_MILES,
    target_distance_pairs=TARGET_DISTANCE_PAIRS,
)

display(distance_check.loc[(distance_check["scenario"]=="baseline") & (distance_check["year"]==1938)].sort_values(["scenario", "year", "origin_district", "dest_district"]))


,scenario,year,origin_district,dest_district,distance_value,distance_km,distance_miles,distance_minutes
147,baseline,1938,BIAŁOSTOCKI,LWÓW (MIASTO),262.132875,421.862100,262.132875,NaN
145,baseline,1938,BIAŁOSTOCKI,WILNO (MIASTO),146.346060,235.521226,146.346060,NaN
146,baseline,1938,KRAKÓW (MIASTO),KIELECKI,80.667420,129.821669,80.667420,NaN
143,baseline,1938,M. ST. WARSZAWA,BIAŁOSTOCKI,116.826181,188.013572,116.826181,NaN
140,baseline,1938,M. ST. WARSZAWA,KRAKÓW (MIASTO),182.337337,293.443590,182.337337,NaN
142,baseline,1938,M. ST. WARSZAWA,LWÓW (MIASTO),249.426731,401.413537,249.426731,NaN
144,baseline,1938,M. ST. WARSZAWA,MORSKI,257.721046,414.761948,257.721046,NaN
141,baseline,1938,M. ST. WARSZAWA,WILNO (MIASTO),253.193253,407.475169,253.193253,NaN
148,baseline,1938,MORSKI,BIELSKO,362.459657,583.322455,362.459657,NaN
149,baseline,1938,POZNAŃ (MIASTO),BIELSKO,233.297278,375.455691,233.297278,NaN


In [ ]:
WARSAW_FOREIGN_CITIES = ["Berlin", "Praha", "Kyiv"]

warsaw_foreign_distance_check = collect_foreign_city_route_distances(
    ma_all=ma_all[ma_all["scenario"] == "baseline"],
    district_to_border_all=district_to_border_all,
    border_connections=border_conn,
    origin_district="M. ST. WARSZAWA",
    foreign_cities=WARSAW_FOREIGN_CITIES,
    compute_in_miles=COMPUTE_IN_MILES,
)

display(
    warsaw_foreign_distance_check.sort_values(["scenario", "year", "foreign_city"])
)


KeyError: 'scenario'

## 6) Export tables

Exports are scenario-specific and include:
- annual total MA,
- annual within-country MA,
- annual foreign-only MA.


In [16]:
for sc in SCENARIOS:
    save_tables(ma_all, OUT_DIR, sc)


Wrote: examples\interwar_poland\market_access\outputs\new_formula\baseline\market_access_annual_baseline.csv
Wrote: examples\interwar_poland\market_access\outputs\new_formula\baseline\market_access_annual_baseline.xlsx
Wrote: examples\interwar_poland\market_access\outputs\new_formula\fixed14\market_access_annual_fixed14.csv
Wrote: examples\interwar_poland\market_access\outputs\new_formula\fixed14\market_access_annual_fixed14.xlsx


## 7) Maps

Generated for each scenario:
- annual maps (1924-1938 as available) for total / domestic / foreign MA,
- change maps for 1938 vs 1924 for all three measures.


In [17]:
for sc in SCENARIOS:
    save_annual_maps(ma_all, sc, "ma_total", "OrRd", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_annual_maps(ma_all, sc, "ma_domestic", "YlGnBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_annual_maps(ma_all, sc, "ma_foreign", "PuRd", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)

    save_change_map_1938_vs_1924(ma_all, sc, "ma_total", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_change_map_1938_vs_1924(ma_all, sc, "ma_domestic", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)
    save_change_map_1938_vs_1924(ma_all, sc, "ma_foreign", "RdBu", PLOTS_DIR, adm_history_plotter, ADM_STATE_DATE, d_city_mapping)

print("Finished maps.")


Finished maps.


## Notes for reviewers

- Partition-border effect is a **binary partition difference** indicator.
- Gdańsk treatment follows domestic distance coefficient.
- The notebook auto-detects available years in `data/distances`; when 1925-1937 files appear, rerunning fills the full annual panel.
- `COMPUTE_IN_MILES=True` converts km-based distances to miles (baseline + foreign connectors).
